# BlEmoRe — CMAFNet v2 (Cross-Modal Attentive Fusion Network)

**Key improvements over CMAFNet v1:**
1. 🧠 **Presence auxiliary head** — 6-way multi-label head predicts which emotions are present, giving richer supervision signal (+0.3 weight to total loss)
2. 🔄 **R-Drop regularization** — Double forward pass; symmetric KL-div between two dropout samples adds consistency pressure with minimal overhead
3. 🎯 **Test-Time Augmentation (TTA)** — 5 forward passes with small feature noise, averaged for more stable probability estimates at inference
4. 🏗️ **2 new architecture variants** — `cmaf_xxl` (128-dim, 8-head, 3-layer) and `cmaf_deep` (96-dim, 8-head, 4-layer) for greater model diversity → 6 variants × 5 folds = 30 models
5. 📈 **More training** — 90 epochs (was 80); SWA starts at epoch 50 (was 60) for more averaged weight coverage
6. 🎲 **Stronger Mixup** — Alpha 0.4 (was 0.3) for harder virtual samples
7. 🏆 **Larger ensemble** — Top-16 checkpoints (was 12), score²-weighted
8. 🧩 **Presence-guided decoding** — Reranks combo candidates by presence head agreement: combo_prob^0.7 × pres_agreement^0.3


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# Copy data from Drive to fast local SSD — adjust paths if yours differ
!cp -rn "/content/drive/MyDrive/blemore_project-backup/data/pre_extracted_train/pre_extracted_train_data/" "/content/pre_extracted_train_data"
!cp -rn "/content/drive/MyDrive/blemore_project-backup/data/pre_extracted_test/pre_extracted_test_data/" "/content/pre_extracted_test_data"


In [3]:
import os, math, random, json, zipfile, copy, time, warnings, gc
from contextlib import nullcontext
from ast import literal_eval
from typing import List, Dict, Optional, Tuple
from itertools import combinations

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTHONHASHSEED"] = "42"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from sklearn.model_selection import GroupKFold

# ── PATHS (adjust if your Drive layout differs) ─────────────────────────
ROOT_TRAIN = "/content/pre_extracted_train_data"
ROOT_TEST  = "/content/pre_extracted_test_data"
OUTPUT_DIR = "/content/drive/MyDrive/blemore_project-backup/blemore_output-cmaf-02"
META_CSV   = "/content/drive/MyDrive/blemore_project-backup/blemore_output-17/train_metadata.csv"

for d in [OUTPUT_DIR, f"{OUTPUT_DIR}/models", f"{OUTPUT_DIR}/epoch_ckpts"]:
    os.makedirs(d, exist_ok=True)

# ── REPRODUCIBILITY ─────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}  VRAM: {p.total_memory/1e9:.1f} GB")

# ── TRAINING HYPER-PARAMETERS ────────────────────────────────────────────
FOLDS           = 5
FOLDS_TO_TRAIN  = [0, 1, 2, 3, 4]
EPOCHS          = 90           # ← was 80: 10 extra epochs of refinement
SWA_START       = 50           # ← was 60: SWA starts earlier, more coverage
SWA_LR          = 2e-5
WARMUP_EPOCHS   = 5
BATCH_SIZE      = 128
NUM_WORKERS     = 0
WEIGHT_DECAY    = 5e-3
GRAD_CLIP       = 1.0
MIXUP_ALPHA     = 0.4          # ← was 0.3: stronger virtual samples
MIX_SAMPLE_MULT = 4.0
ENSEMBLE_TOPK   = 16           # ← was 12: more model diversity
TTA_PASSES      = 5            # NEW: forward passes per sample at inference
RDROP_ALPHA     = 0.5          # NEW: R-Drop KL-div weight

# ── ARCHITECTURE VARIANTS ────────────────────────────────────────────────
# v1 kept: cmaf_s, cmaf_m, cmaf_l, cmaf_xl
# v2 adds:  cmaf_xxl (128-dim, 8-head), cmaf_deep (96-dim, 8-head, 4-layer)
MODEL_VARIANTS = [
    dict(name="cmaf_s",    mod_proj=64,  n_heads=4, n_layers=2, dropout=0.45, lr=4e-4),
    dict(name="cmaf_m",    mod_proj=96,  n_heads=4, n_layers=2, dropout=0.40, lr=3e-4),
    dict(name="cmaf_l",    mod_proj=64,  n_heads=4, n_layers=3, dropout=0.45, lr=3e-4),
    dict(name="cmaf_xl",   mod_proj=96,  n_heads=4, n_layers=3, dropout=0.50, lr=3e-4),
    dict(name="cmaf_xxl",  mod_proj=128, n_heads=8, n_layers=3, dropout=0.45, lr=2e-4),  # NEW
    dict(name="cmaf_deep", mod_proj=96,  n_heads=8, n_layers=4, dropout=0.50, lr=2e-4),  # NEW
]
print(f"Variants: {[v['name'] for v in MODEL_VARIANTS]}")
print(f"Total models to train: {len(MODEL_VARIANTS) * len(FOLDS_TO_TRAIN)}")


Device: cpu
Variants: ['cmaf_s', 'cmaf_m', 'cmaf_l', 'cmaf_xl', 'cmaf_xxl', 'cmaf_deep']
Total models to train: 30


In [4]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 4 — Emotion taxonomy & 21-class label space           ║
# ╚══════════════════════════════════════════════════════════════╝
EMO_CODES  = ['ang', 'disg', 'fea', 'hap', 'sad', 'neu']
EMO2IDX    = {e: i for i, e in enumerate(EMO_CODES)}
N_EMO      = len(EMO_CODES)

SINGLE_COMBOS  = [(e,) for e in EMO_CODES]                              # 6 singles
BLENDED_COMBOS = [tuple(sorted(p)) for p in combinations(EMO_CODES, 2)] # 15 blends
ALL_COMBOS     = SINGLE_COMBOS + BLENDED_COMBOS                         # 21 total
N_COMBO        = len(ALL_COMBOS)
COMBO2IDX      = {c: i for i, c in enumerate(ALL_COMBOS)}
N_SAL          = 3   # salience classes: {0=first_dom 70/30, 1=equal 50/50, 2=second_dom 30/70}

print(f"N_COMBO={N_COMBO}")
print("Singles: ", ALL_COMBOS[:6])
print("Blended (first 5):", ALL_COMBOS[6:11], "...")


N_COMBO=21
Singles:  [('ang',), ('disg',), ('fea',), ('hap',), ('sad',), ('neu',)]
Blended (first 5): [('ang', 'disg'), ('ang', 'fea'), ('ang', 'hap'), ('ang', 'sad'), ('ang', 'neu')] ...


In [5]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 5 — Feature utilities                                 ║
# ╚══════════════════════════════════════════════════════════════╝
def _pool(arr) -> Optional[np.ndarray]:
    """Temporal mean+max pooling over frame axis."""
    try: arr = np.asarray(arr, dtype=np.float32)
    except: return None
    arr = np.nan_to_num(arr, nan=0., posinf=0., neginf=0.)
    if arr.ndim == 0 or arr.size == 0: return None
    if arr.ndim == 1: return np.concatenate([arr, arr])  # single frame: replicate
    x = arr.reshape(-1, arr.shape[-1])
    return np.concatenate([x.mean(0), x.max(0)])

def _fix_dim(v: np.ndarray, d: int) -> np.ndarray:
    v = np.asarray(v, dtype=np.float32).reshape(-1)
    if v.shape[0] == d: return v
    if v.shape[0]  > d: return v[:d]
    return np.concatenate([v, np.zeros(d - v.shape[0], dtype=np.float32)])

def _find_folder(root: str, mod: str) -> Optional[str]:
    """Try common name variants to locate a modality folder."""
    candidates = [
        mod,
        mod.replace('_train', '_test'),
        mod.replace('_test',  '_train'),
        mod.replace('_train', ''),
        mod.replace('_test',  ''),
        mod + '_train',
    ]
    for c in candidates:
        if c and os.path.isdir(os.path.join(root, c)):
            return c
    return None

def _raw_pool(root: str, mod_dir: str, fname: str) -> Optional[np.ndarray]:
    base = os.path.splitext(os.path.basename(fname))[0]
    for path in [f"{root}/{mod_dir}/{base}.npy",
                 f"{root}/{mod_dir}/{base}.mov.npy",
                 f"{root}/{mod_dir}/{fname}.npy"]:
        if not os.path.exists(path): continue
        try:
            p = _pool(np.load(path, allow_pickle=True))
            if p is not None: return p
        except Exception as e:
            warnings.warn(f"[_raw_pool] {path}: {e}")
    return None

def load_feature(root, mod_dir, fname, dim):
    p = _raw_pool(root, mod_dir, fname)
    return _fix_dim(p, dim) if p is not None else None

def _parse_vec(x, n):
    out = np.zeros(n, dtype=np.float32)
    if isinstance(x, (list, tuple, np.ndarray)):
        a = np.asarray(x, dtype=np.float32).reshape(-1)
        out[:min(n, len(a))] = a[:min(n, len(a))]
    elif isinstance(x, str):
        try: return _parse_vec(literal_eval(x), n)
        except: pass
    return out


In [6]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 6 — Modality discovery, normalization stats, cache    ║
# ╚══════════════════════════════════════════════════════════════╝

ALL_POSSIBLE_MODS = [
    'VideoMAEv2_train', 'VideoSwinTransformer_train',
    'OpenFace2_train', 'OpenFace3_train',
    'WavLM_large_train', 'HuBERT_large_train',
    'ImageBind_train', 'CLIP_train',
    'DINOv2_train', 'DINOv3_train',
    'HiCMAE_train',
]

print("Scanning TRAIN root for available modalities:")
selected_mods, seen_dirs = [], set()
for m in ALL_POSSIBLE_MODS:
    folder = _find_folder(ROOT_TRAIN, m)
    if folder and folder not in seen_dirs:
        seen_dirs.add(folder)
        selected_mods.append(m)
        print(f"  ✓ {m:30s} -> {folder}")

assert selected_mods, "No modalities found! Check ROOT_TRAIN."
print(f"\n→ {len(selected_mods)} modalities found: {selected_mods}")

first_folder  = _find_folder(ROOT_TRAIN, selected_mods[0])
train_samples = sorted(
    os.path.splitext(f)[0]
    for f in os.listdir(f"{ROOT_TRAIN}/{first_folder}")
    if f.lower().endswith('.npy')
)
print(f"Training samples: {len(train_samples)}")

def compute_stats(root, modalities, samples, n=500):
    stats = {}
    for mod in modalities:
        mod_dir = _find_folder(root, mod)
        if not mod_dir: continue
        probe_dim = next(
            (p.shape[0] for s in samples[:n]
             if (p := _raw_pool(root, mod_dir, s)) is not None), None)
        if probe_dim is None: continue
        feats = [f for s in samples[:n]
                 if (f := load_feature(root, mod_dir, s, probe_dim)) is not None]
        if not feats: continue
        mat = np.stack(feats).astype(np.float32)
        std = mat.std(0); std[std < 1e-6] = 1.
        stats[mod] = {"dim": probe_dim, "mean": mat.mean(0), "std": std}
        print(f"  {mod}: dim={probe_dim}  n={len(feats)}")
    print(f"Total feat dim: {sum(v['dim'] for v in stats.values())}")
    return stats

cache_path = f"{OUTPUT_DIR}/modality_stats.npy"
if os.path.exists(cache_path):
    stats    = np.load(cache_path, allow_pickle=True).item()
    new_mods = [m for m in selected_mods if m not in stats]
    if new_mods:
        print(f"Computing stats for new mods: {new_mods}")
        stats.update(compute_stats(ROOT_TRAIN, new_mods, train_samples))
        np.save(cache_path, stats, allow_pickle=True)
    print("Loaded stats:", {m: v['dim'] for m, v in stats.items()})
else:
    stats = compute_stats(ROOT_TRAIN, selected_mods, train_samples)
    np.save(cache_path, stats, allow_pickle=True)

selected_mods = [m for m in selected_mods if m in stats]
print(f"\nFinal modalities ({len(selected_mods)}): {selected_mods}")
total_feat_dim = sum(stats[m]['dim'] for m in selected_mods)
print(f"Total feature dimension: {total_feat_dim}")

def build_ram_cache(root_dir, modalities, stats, filenames, desc="Cache"):
    mod_dirs = {m: _find_folder(root_dir, m) or m for m in modalities}
    dims  = {m: int(stats[m]['dim']) for m in modalities}
    means = {m: np.asarray(stats[m]['mean'], dtype=np.float32) for m in modalities}
    stds  = {m: np.asarray(stats[m]['std'],  dtype=np.float32) for m in modalities}
    cache = {}
    for fname in tqdm(filenames, desc=desc):
        for m in modalities:
            feat = load_feature(root_dir, mod_dirs[m], fname, dims[m])
            if feat is not None:
                cache[(m, fname)] = (feat - means[m]) / (stds[m] + 1e-6)
    n_total = len(filenames) * len(modalities)
    hit_pct = 100 * len(cache) / n_total if n_total else 0
    ram_mb  = sum(v.nbytes for v in cache.values()) / 1e6
    print(f"  {desc}: {len(cache)}/{n_total} ({hit_pct:.1f}%)  RAM: {ram_mb:.1f} MB")
    return cache

print("\nBuilding TRAIN cache ...")
train_cache = build_ram_cache(ROOT_TRAIN, selected_mods, stats, train_samples, "TRAIN")

test_folder  = _find_folder(ROOT_TEST, selected_mods[0])
test_samples = sorted(
    os.path.splitext(f)[0]
    for f in os.listdir(f"{ROOT_TEST}/{test_folder}")
    if f.lower().endswith('.npy')
)
print("Building TEST cache ...")
test_cache = build_ram_cache(ROOT_TEST, selected_mods, stats, test_samples, "TEST")
print(f"Test samples: {len(test_samples)}")


Scanning TRAIN root for available modalities:
  ✓ VideoMAEv2_train               -> VideoMAEv2_train
  ✓ VideoSwinTransformer_train     -> VideoSwinTransformer_train
  ✓ WavLM_large_train              -> WavLM_large_train
  ✓ HuBERT_large_train             -> HuBERT_large_train

→ 4 modalities found: ['VideoMAEv2_train', 'VideoSwinTransformer_train', 'WavLM_large_train', 'HuBERT_large_train']
Training samples: 2456
Loaded stats: {'VideoMAEv2_train': 2816, 'VideoSwinTransformer_train': 1536, 'WavLM_large_train': 2048, 'HuBERT_large_train': 2048}

Final modalities (4): ['VideoMAEv2_train', 'VideoSwinTransformer_train', 'WavLM_large_train', 'HuBERT_large_train']
Total feature dimension: 8448

Building TRAIN cache ...


TRAIN: 100%|██████████| 2456/2456 [00:49<00:00, 49.42it/s]


  TRAIN: 9824/9824 (100.0%)  RAM: 83.0 MB
Building TEST cache ...


TEST: 100%|██████████| 594/594 [00:10<00:00, 58.02it/s]

  TEST: 2267/2376 (95.4%)  RAM: 19.2 MB
Test samples: 594


In [7]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 7 — Metadata with 21-class + 3-class salience labels  ║
# ╚══════════════════════════════════════════════════════════════╝
def _is_miss(x):
    if x is None: return True
    try: return pd.isna(x)
    except: return False

def _emo(x):
    if _is_miss(x): return None
    s = str(x).strip().lower()
    return s if s in EMO2IDX else None

def _fl(x):
    if _is_miss(x): return None
    try: return float(x)
    except: return None

def build_sample_labels(row) -> Dict:
    e1 = _emo(row.get('emotion_1')) or 'neu'
    e2 = _emo(row.get('emotion_2'))
    s1 = _fl(row.get('emotion_1_salience'))
    s2 = _fl(row.get('emotion_2_salience'))
    is_blend = (int(row.get('mix', 0)) == 1) or (e2 in EMO2IDX)

    if not is_blend or e2 not in EMO2IDX:
        combo     = (e1,)
        combo_idx = COMBO2IDX.get(combo, 0)
        sal_idx   = 0
        sal_mask  = 0.
        pres_list = [1. if e == e1 else 0. for e in EMO_CODES]
    else:
        ea, eb    = tuple(sorted([e1, e2]))
        combo     = (ea, eb)
        combo_idx = COMBO2IDX.get(combo, N_COMBO - 1)
        if s1 is None or s2 is None: s1, s2 = 50., 50.
        t = s1 + s2; s1, s2 = s1 / t * 100, s2 / t * 100
        sal_ea    = s1 if e1 == ea else s2
        sal_idx   = 0 if sal_ea >= 65 else (2 if sal_ea <= 35 else 1)
        sal_mask  = 1.
        pres_list = [1. if e in (ea, eb) else 0. for e in EMO_CODES]

    return dict(combo_idx=combo_idx, sal_idx=sal_idx,
                sal_mask=float(is_blend), pres_list=pres_list, is_blend=int(is_blend))

def load_metadata(csv_path):
    df = pd.read_csv(csv_path)
    if 'filename' not in df.columns and 'fname' in df.columns:
        df = df.rename(columns={'fname': 'filename'})
    label_rows = [build_sample_labels(r) for _, r in df.iterrows()]
    for k in ['combo_idx', 'sal_idx', 'sal_mask', 'pres_list', 'is_blend']:
        df[k] = [lr[k] for lr in label_rows]
    if 'fold' not in df.columns or df['fold'].isnull().all():
        gc_col = 'video_id' if 'video_id' in df.columns else 'filename'
        df['fold'] = -1
        for fi, (_, vi) in enumerate(
                GroupKFold(n_splits=FOLDS).split(df, groups=df[gc_col].values)):
            df.loc[vi, 'fold'] = fi
    else:
        df['fold'] = df['fold'].astype(int)
    return df

meta_df = load_metadata(META_CSV)
n_b = int(meta_df['is_blend'].sum())
print(f"Rows: {len(meta_df)}  Blended: {n_b} ({100*n_b/len(meta_df):.1f}%)")
combo_counts = meta_df['combo_idx'].value_counts().sort_index()
print(f"Unique combos used: {len(combo_counts)}/21  Least common: {combo_counts.min()}")


Rows: 2456  Blended: 1326 (54.0%)
Unique combos used: 16/21  Least common: 126


In [8]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 8 — Dataset                                           ║
# ╚══════════════════════════════════════════════════════════════╝
class BlEmoDataset(Dataset):
    def __init__(self, meta_df, modalities, fold, split, stats,
                 ram_cache, modality_drop=0.0, feat_noise=0.01):
        assert split in ('train', 'val')
        df = (meta_df[meta_df['fold'] != fold] if split == 'train'
              else meta_df[meta_df['fold'] == fold])
        self.df       = df.reset_index(drop=True)
        self.mods     = modalities
        self.is_train = (split == 'train')
        self.mod_drop = modality_drop
        self.noise    = feat_noise
        self.cache    = ram_cache
        self.dims     = [int(stats[m]['dim']) for m in modalities]
        self.feat_len = sum(self.dims)
        print(f"  [{split} fold={fold}] n={len(self.df)}  feat_len={self.feat_len}")

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        fname  = str(row['filename'])
        drop_m = (random.choice(self.mods)
                  if self.is_train and random.random() < self.mod_drop else None)
        feats  = []
        for m, dim in zip(self.mods, self.dims):
            if m == drop_m:
                vec = np.zeros(dim, dtype=np.float32)
            else:
                vec = self.cache.get((m, fname))
                if vec is None:
                    vec = np.zeros(dim, dtype=np.float32)
                else:
                    vec = vec.copy()
                    if self.is_train and self.noise > 0:
                        vec += np.random.randn(dim).astype(np.float32) * self.noise
            feats.append(vec.astype(np.float32))
        return {
            "features"  : torch.from_numpy(np.concatenate(feats)),
            "combo_idx" : torch.tensor(int(row['combo_idx']),  dtype=torch.long),
            "sal_idx"   : torch.tensor(int(row['sal_idx']),    dtype=torch.long),
            "sal_mask"  : torch.tensor(float(row['sal_mask']), dtype=torch.float32),
            "pres_list" : torch.tensor(_parse_vec(row['pres_list'], N_EMO), dtype=torch.float32),
            "is_blend"  : torch.tensor(int(row['is_blend']),   dtype=torch.long),
            "fname"     : fname,
        }


class BlEmoTestDataset(Dataset):
    def __init__(self, filenames, modalities, stats, ram_cache):
        self.filenames = filenames; self.mods = modalities
        self.dims  = [int(stats[m]['dim']) for m in modalities]
        self.cache = ram_cache
    def __len__(self): return len(self.filenames)
    def __getitem__(self, idx):
        fname = self.filenames[idx]; feats = []
        for m, dim in zip(self.mods, self.dims):
            vec = self.cache.get((m, fname))
            if vec is None: vec = np.zeros(dim, dtype=np.float32)
            feats.append(vec.astype(np.float32))
        return {"features": torch.from_numpy(np.concatenate(feats)), "fname": fname}


In [9]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 9 — CMAFNet v2: Cross-Modal Attentive Fusion          ║
# ║                                                              ║
# ║  NEW vs v1:                                                  ║
# ║  • pres_head: 6-way multi-label presence prediction.        ║
# ║    Gives the model an auxiliary signal about WHICH emotions  ║
# ║    are present before it commits to a 21-class combo.       ║
# ║    Supervision is cheap (pres_list already in the dataset)  ║
# ║    but the gradient signal meaningfully regularizes the      ║
# ║    shared representation.                                    ║
# ╚══════════════════════════════════════════════════════════════╝

class ModalityEncoder(nn.Module):
    """Project one modality's pooled features to mod_proj-dim token."""
    def __init__(self, in_dim: int, mod_proj: int, dropout: float):
        super().__init__()
        mid = mod_proj * 2
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, mid),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mid, mod_proj),
            nn.GELU(),
            nn.LayerNorm(mod_proj),
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class CrossModalBlock(nn.Module):
    """Pre-norm transformer block with self-attention across all modality tokens."""
    def __init__(self, d: int, n_heads: int, dropout: float):
        super().__init__()
        self.ln1  = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.ln2  = nn.LayerNorm(d)
        self.ff   = nn.Sequential(
            nn.Linear(d, d * 4), nn.GELU(), nn.Dropout(dropout), nn.Linear(d * 4, d)
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.ln1(x)
        h, _ = self.attn(h, h, h)
        x = x + self.drop(h)
        h = self.ln2(x)
        x = x + self.drop(self.ff(h))
        return x


class CMAFNet(nn.Module):
    """
    CMAFNet v2 — adds presence auxiliary head.

    Forward returns THREE tensors:
        combo_logits  [B, 21]  — main 21-class emotion combo classification
        sal_logits    [B, 3]   — salience classification (blended only)
        pres_logits   [B, 6]   — presence prediction (multi-label, BCEWithLogits)

    The presence head shares the fused representation with the combo/sal
    heads but has its own linear layer. Its gradient flows back through
    the shared backbone, effectively regularising it toward learning
    which individual emotions are active — a cleaner inductive bias than
    raw 21-class supervision alone.
    """
    def __init__(self, stats, modalities,
                 mod_proj: int = 64, n_heads: int = 4,
                 n_layers: int = 2, dropout: float = 0.45):
        super().__init__()
        self.mods       = modalities
        self.split_dims = [int(stats[m]['dim']) for m in modalities]
        n_mod           = len(modalities)
        d               = mod_proj
        head_dim        = d * 2

        self.encoders = nn.ModuleList([
            ModalityEncoder(dim_i, d, dropout) for dim_i in self.split_dims
        ])

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        self.pos_emb = nn.Parameter(torch.zeros(1, n_mod + 1, d))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)

        self.blocks   = nn.ModuleList([CrossModalBlock(d, n_heads, dropout)
                                        for _ in range(n_layers)])
        self.final_ln = nn.LayerNorm(d)
        self.attn_pool = nn.Linear(d, 1)

        fuse_dim = d * 2
        self.pre_head = nn.Sequential(
            nn.LayerNorm(fuse_dim),
            nn.Linear(fuse_dim, head_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(head_dim),
        )

        # ── Three output heads ───────────────────────────────────────
        self.combo_head = nn.Linear(head_dim, N_COMBO)   # 21-class
        self.sal_head   = nn.Linear(head_dim, N_SAL)     # 3-class
        self.pres_head  = nn.Linear(head_dim, N_EMO)     # NEW: 6-way multi-label

        for head in [self.combo_head, self.sal_head, self.pres_head]:
            nn.init.trunc_normal_(head.weight, std=0.02)
            nn.init.zeros_(head.bias)

    def forward(self, flat: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        B = flat.size(0)
        parts = torch.split(flat, self.split_dims, dim=1)

        mod_tokens = torch.stack(
            [self.encoders[i](p) for i, p in enumerate(parts)], dim=1
        )

        cls    = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, mod_tokens], dim=1)
        tokens = tokens + self.pos_emb

        for block in self.blocks:
            tokens = block(tokens)
        tokens = self.final_ln(tokens)

        cls_out = tokens[:, 0]
        mod_out = tokens[:, 1:]

        attn_w  = F.softmax(self.attn_pool(mod_out), dim=1)
        pooled  = (attn_w * mod_out).sum(1)

        fused   = torch.cat([cls_out, pooled], dim=-1)
        h       = self.pre_head(fused)
        return self.combo_head(h), self.sal_head(h), self.pres_head(h)


def count_params(m):
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  Params: {n/1e6:.3f}M")
    return n

# Sanity check
_test_dims = {m: stats[m] for m in selected_mods}
_m = CMAFNet(_test_dims, selected_mods, mod_proj=64, n_heads=4, n_layers=2)
print("CMAFNet v2 (cmaf_s) sanity check:")
count_params(_m)
_x = torch.zeros(4, sum(stats[m]['dim'] for m in selected_mods))
_c, _s, _p = _m(_x)
print(f"  combo_logits: {_c.shape}  sal_logits: {_s.shape}  pres_logits: {_p.shape}")
del _m, _x, _c, _s, _p


CMAFNet v2 (cmaf_s) sanity check:
  Params: 1.254M
  combo_logits: torch.Size([4, 21])  sal_logits: torch.Size([4, 3])  pres_logits: torch.Size([4, 6])


In [10]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 10 — Loss functions                                   ║
# ║  NEW v2: presence BCE loss + R-Drop KL-div regularization   ║
# ╚══════════════════════════════════════════════════════════════╝

def focal_ce(logits, targets, gamma=2.0, weight=None, label_smoothing=0.0):
    """Focal cross-entropy: down-weights easy examples, focuses on hard ones."""
    ce = F.cross_entropy(logits, targets, weight=weight,
                         label_smoothing=label_smoothing, reduction='none')
    pt = torch.exp(-ce.detach())
    return ((1.0 - pt) ** gamma * ce).mean()


def soft_cross_entropy(logits, soft_targets):
    """Cross-entropy with soft (mixup) targets."""
    log_p = F.log_softmax(logits, dim=-1)
    return -(soft_targets * log_p).sum(-1).mean()


def make_soft_labels(y, n_classes, smoothing=0.0):
    """One-hot + optional label smoothing, suitable for mixup."""
    B = y.size(0)
    s = torch.full((B, n_classes), smoothing / max(n_classes - 1, 1), device=y.device)
    s.scatter_(1, y.unsqueeze(1), 1.0 - smoothing)
    return s


def rdrop_loss(logits1: torch.Tensor, logits2: torch.Tensor,
               alpha: float = RDROP_ALPHA) -> torch.Tensor:
    """
    R-Drop: symmetric KL-divergence between two dropout samples.

    During training the model is run TWICE on the same input with
    different random dropout masks.  Minimising the KL-div between the
    two output distributions acts as a consistency regulariser — the
    model learns predictions that are robust to which neurons are dropped.

    alpha controls the strength; 0.5 is a conservative starting point
    that avoids over-shadowing the CE terms.
    """
    p = F.log_softmax(logits1, dim=-1)
    q = F.log_softmax(logits2, dim=-1)
    kl_pq = F.kl_div(p, q.exp().detach(), reduction='batchmean')
    kl_qp = F.kl_div(q, p.exp().detach(), reduction='batchmean')
    return alpha * 0.5 * (kl_pq + kl_qp)


def compute_combo_weights(df) -> torch.Tensor:
    """Inverse-frequency class weights for the 21 combo classes."""
    counts = np.zeros(N_COMBO, dtype=np.float32)
    for idx in df['combo_idx'].values:
        counts[int(idx)] += 1
    counts = np.maximum(counts, 1)
    w = (len(df) / (N_COMBO * counts)).clip(0.5, 10.)
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


def compute_loss(combo_logits, sal_logits, pres_logits,
                 combo_targets, sal_targets, sal_mask, pres_targets,
                 combo_soft=None, sal_soft=None,
                 combo_weights=None, gamma=2.0, label_smoothing=0.05):
    """
    Multi-task loss (v2):
      combo_loss  — focal CE for 21-class classification          (weight 1.0)
      sal_loss    — focal CE for 3-class salience (blends only)   (weight 0.7)
      pres_loss   — BCE for 6-way multi-label presence            (weight 0.3)

    The presence loss is an auxiliary signal that shares the backbone:
    it nudges the representation toward identifying individual emotions
    before committing to a 21-class combo, improving generalisation.
    """
    # ── Combo loss ──────────────────────────────────────────────────────
    if combo_soft is not None:
        combo_loss = soft_cross_entropy(combo_logits, combo_soft)
    else:
        combo_loss = focal_ce(combo_logits, combo_targets,
                               gamma=gamma, weight=combo_weights,
                               label_smoothing=label_smoothing)

    # ── Salience loss (blended samples only) ────────────────────────────
    if sal_mask.sum() > 0:
        blend_idx = sal_mask > 0.5
        if sal_soft is not None:
            sal_loss = soft_cross_entropy(sal_logits[blend_idx], sal_soft[blend_idx])
        else:
            sal_loss = focal_ce(sal_logits[blend_idx], sal_targets[blend_idx],
                                 gamma=1.5, label_smoothing=label_smoothing)
    else:
        sal_loss = combo_logits.new_tensor(0.)

    # ── Presence loss (multi-label BCE) — NEW ───────────────────────────
    pres_loss = F.binary_cross_entropy_with_logits(pres_logits, pres_targets)

    total = combo_loss + 0.7 * sal_loss + 0.3 * pres_loss
    return total, combo_loss, sal_loss, pres_loss


In [11]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 11 — Decoding & official metrics                      ║
# ║  NEW v2: decode_with_presence — presence-guided reranking   ║
# ╚══════════════════════════════════════════════════════════════╝
def decode_sample(combo_idx: int, sal_idx: int) -> List[Dict]:
    combo = ALL_COMBOS[combo_idx]
    if len(combo) == 1:
        return [{'emotion': combo[0], 'salience': 100.}]
    ea, eb = combo[0], combo[1]
    if   sal_idx == 0: sa, sb = 70., 30.
    elif sal_idx == 1: sa, sb = 50., 50.
    else:              sa, sb = 30., 70.
    return [{'emotion': ea, 'salience': sa}, {'emotion': eb, 'salience': sb}]


def decode_batch(combo_probs, sal_probs, fnames):
    """Simple argmax decoding (used during training validation for speed)."""
    pred = {}
    for i, fn in enumerate(fnames):
        key       = fn if fn.endswith('.mov') else fn + '.mov'
        pred[key] = decode_sample(int(np.argmax(combo_probs[i])),
                                   int(np.argmax(sal_probs[i])))
    return pred


def decode_with_presence(combo_probs, sal_probs, pres_probs, fnames):
    """
    Presence-guided combo reranking (NEW in v2).

    For each sample we compute a reranked score:
        reranked[ci] = combo_prob[ci]^0.7 × pres_agreement[ci]^0.3

    pres_agreement[ci] = geometric mean of presence probs for emotions
    in that combo.  This softly boosts combos whose emotion constituents
    the presence head is confident about, and penalises combos that
    include emotions the presence head thinks are absent.

    The 0.7 / 0.3 split keeps the combo classifier primary while letting
    the presence head act as a tie-breaker — empirically this is more
    stable than a 50/50 split.
    """
    pred = {}
    for i, fn in enumerate(fnames):
        combo_scores = combo_probs[i].copy().astype(np.float64)
        for ci, combo in enumerate(ALL_COMBOS):
            # Geometric mean of presence probs for emotions IN this combo
            pres_agree = float(np.prod(
                [0.5 + 0.5 * float(pres_probs[i, EMO2IDX[e]]) for e in combo]
            ))
            combo_scores[ci] = (combo_scores[ci] ** 0.7) * (pres_agree ** 0.3)
        best_combo = int(np.argmax(combo_scores))
        key        = fn if fn.endswith('.mov') else fn + '.mov'
        pred[key]  = decode_sample(best_combo, int(np.argmax(sal_probs[i])))
    return pred


def build_truth(df):
    truth = {}
    for _, row in df.iterrows():
        fn  = str(row['filename'])
        key = fn if fn.endswith('.mov') else fn + '.mov'
        truth[key] = decode_sample(int(row['combo_idx']), int(row['sal_idx']))
    return truth

def official_scores(pred_map: Dict, df: pd.DataFrame) -> Tuple[float, float, float]:
    truth  = build_truth(df)
    acc_p, acc_s = [], []
    for key, gt in truth.items():
        pr   = pred_map.get(key, [])
        gt_e = sorted(x['emotion'] for x in gt)
        pr_e = sorted(x['emotion'] for x in pr)
        p_ok = int(gt_e == pr_e)
        acc_p.append(p_ok)
        if len(gt) == 2:
            if p_ok:
                gt_d = {x['emotion']: round(float(x['salience'])) for x in gt}
                pr_d = {x['emotion']: round(float(x['salience'])) for x in pr}
                acc_s.append(int(gt_d == pr_d))
            else:
                acc_s.append(0)
    a = float(np.mean(acc_p)) if acc_p else 0.
    b = float(np.mean(acc_s)) if acc_s else 0.
    return a, b, 0.5 * (a + b)


In [12]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 12 — Training utilities                               ║
# ║  NEW v2: R-Drop in train_epoch; presence probs in val_epoch  ║
# ╚══════════════════════════════════════════════════════════════╝
def make_sample_weights(df):
    mix = df['sal_mask'].values.astype(np.float64)
    return torch.as_tensor(np.where(mix > 0.5, MIX_SAMPLE_MULT, 1.), dtype=torch.double)

def get_amp():
    if DEVICE.type == 'cuda':
        try: return torch.amp.autocast('cuda')
        except: return torch.cuda.amp.autocast()
    return nullcontext()

def get_scaler():
    if DEVICE.type == 'cuda':
        try: return torch.amp.GradScaler('cuda')
        except: return torch.cuda.amp.GradScaler(enabled=True)
    return None

def build_cosine_schedule(optimizer, warmup_steps, total_steps, min_lr=0.05):
    def fn(step):
        if step < warmup_steps: return (step + 1) / warmup_steps
        p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(min_lr, 0.5 * (1. + math.cos(math.pi * p)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, fn)

def make_loader(ds, shuffle=False, sampler=None):
    return DataLoader(ds, BATCH_SIZE, shuffle=shuffle, sampler=sampler,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'))


def apply_mixup(feats, combo_idx, sal_idx, sal_mask):
    """Feature-space Mixup — creates virtual samples between real ones."""
    if MIXUP_ALPHA <= 0 or random.random() > 0.5:
        return feats, None, None, sal_mask

    B    = feats.size(0)
    lam  = torch.from_numpy(
        np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA, size=B).astype(np.float32)
    ).to(feats.device)
    perm = torch.randperm(B, device=feats.device)

    mixed = lam[:, None] * feats + (1 - lam[:, None]) * feats[perm]

    s1c = make_soft_labels(combo_idx,       N_COMBO, smoothing=0.05)
    s2c = make_soft_labels(combo_idx[perm], N_COMBO, smoothing=0.05)
    combo_soft = lam[:, None] * s1c + (1 - lam[:, None]) * s2c

    s1s = make_soft_labels(sal_idx,       N_SAL, smoothing=0.05)
    s2s = make_soft_labels(sal_idx[perm], N_SAL, smoothing=0.05)
    sal_soft = lam[:, None] * s1s + (1 - lam[:, None]) * s2s

    mixed_sal_mask = torch.max(sal_mask, sal_mask[perm])
    return mixed, combo_soft, sal_soft, mixed_sal_mask


def train_epoch(model, ds_tr, opt, sched, scaler, combo_w):
    """
    Training step with R-Drop regularization (NEW in v2).

    R-Drop: run the model TWICE on the same (possibly mixed) input with
    different dropout masks.  Average the two CE losses and add a
    symmetric KL-div term.  This is applied to every batch and costs
    roughly one extra forward pass (~50% overhead), but delivers
    consistent regularisation gains on small datasets.
    """
    model.train()
    total, n = 0., 0
    sampler  = WeightedRandomSampler(make_sample_weights(ds_tr.df), len(ds_tr), replacement=True)
    loader   = make_loader(ds_tr, sampler=sampler)
    opt.zero_grad(set_to_none=True)

    for batch in loader:
        feats    = batch['features'].to(DEVICE, non_blocking=True)
        c_idx    = batch['combo_idx'].to(DEVICE, non_blocking=True)
        s_idx    = batch['sal_idx'].to(DEVICE,   non_blocking=True)
        s_mask   = batch['sal_mask'].to(DEVICE,  non_blocking=True)
        pres_tgt = batch['pres_list'].to(DEVICE, non_blocking=True)

        mixed, combo_soft, sal_soft, eff_mask = apply_mixup(feats, c_idx, s_idx, s_mask)

        with get_amp():
            # ── First forward pass ──────────────────────────────────
            cl1, sl1, pl1 = model(mixed)
            loss1, *_ = compute_loss(cl1, sl1, pl1,
                                      c_idx, s_idx, eff_mask, pres_tgt,
                                      combo_soft=combo_soft, sal_soft=sal_soft,
                                      combo_weights=combo_w)
            # ── Second forward pass (different dropout mask) ────────
            cl2, sl2, pl2 = model(mixed)
            loss2, *_ = compute_loss(cl2, sl2, pl2,
                                      c_idx, s_idx, eff_mask, pres_tgt,
                                      combo_soft=combo_soft, sal_soft=sal_soft,
                                      combo_weights=combo_w)
            # ── R-Drop: symmetric KL-div between the two passes ─────
            rd = rdrop_loss(cl1, cl2, RDROP_ALPHA)
            loss = 0.5 * (loss1 + loss2) + rd

        if scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt); scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        if sched: sched.step()
        opt.zero_grad(set_to_none=True)
        total += loss.item() * feats.size(0); n += feats.size(0)
    return total / max(n, 1)


@torch.no_grad()
def val_epoch(model, ds_va, tta_passes: int = 1, tta_noise: float = 0.015):
    """
    Validation inference.

    tta_passes=1  → standard single-pass (used during training for speed)
    tta_passes=TTA_PASSES → TTA: average several noisy passes (used in
                             run_fold final eval + ensemble inference)
    """
    model.eval()
    loader = make_loader(ds_va, shuffle=False)
    cp, sp, pp, fns = [], [], [], []
    for batch in loader:
        feats = batch['features'].to(DEVICE, non_blocking=True)
        batch_cp = batch_sp = batch_pp = None
        for t in range(tta_passes):
            noisy = feats if t == 0 else feats + torch.randn_like(feats) * tta_noise
            with get_amp():
                cl, sl, pl = model(noisy)
            bc = torch.softmax(cl, dim=1)
            bs = torch.softmax(sl, dim=1)
            bp = torch.sigmoid(pl)
            batch_cp = bc if batch_cp is None else batch_cp + bc
            batch_sp = bs if batch_sp is None else batch_sp + bs
            batch_pp = bp if batch_pp is None else batch_pp + bp
        cp.append((batch_cp / tta_passes).cpu().numpy())
        sp.append((batch_sp / tta_passes).cpu().numpy())
        pp.append((batch_pp / tta_passes).cpu().numpy())
        fns.extend(batch['fname'])
    return np.vstack(cp), np.vstack(sp), np.vstack(pp), fns


In [13]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 13 — run_fold with SWA (updated for 3-head model)     ║
# ╚══════════════════════════════════════════════════════════════╝
def run_fold(fold, cfg, meta_df, stats, ram_cache, epochs=EPOCHS):
    name = cfg['name']
    print(f"\n{'─'*60}\nFold {fold} | {name}\n{'─'*60}")

    ds_tr = BlEmoDataset(meta_df, selected_mods, fold, 'train', stats,
                          ram_cache, modality_drop=0.15, feat_noise=0.01)
    ds_va = BlEmoDataset(meta_df, selected_mods, fold, 'val',   stats,
                          ram_cache, modality_drop=0.,   feat_noise=0.)

    model = CMAFNet(
        stats, selected_mods,
        mod_proj=cfg['mod_proj'], n_heads=cfg['n_heads'],
        n_layers=cfg['n_layers'], dropout=cfg['dropout'],
    ).to(DEVICE)
    count_params(model)
    swa_model = AveragedModel(model)

    steps_per_ep = max(1, math.ceil(len(ds_tr) / BATCH_SIZE))
    total_steps  = epochs * steps_per_ep
    warmup_steps = WARMUP_EPOCHS * steps_per_ep

    opt     = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=WEIGHT_DECAY)
    sched   = build_cosine_schedule(opt, warmup_steps, total_steps)
    swa_s   = SWALR(opt, swa_lr=SWA_LR, anneal_epochs=5)
    scaler  = get_scaler()
    combo_w = compute_combo_weights(ds_tr.df)

    best_score, best_state = -1., None
    hist, start_ep = [], 1

    best_ckpt = f"{OUTPUT_DIR}/models/{name}_fold{fold}.pt"
    if os.path.exists(best_ckpt):
        ck = torch.load(best_ckpt, map_location=DEVICE, weights_only=False)
        try:
            model.load_state_dict(ck['model_state'], strict=False)
            best_score = ck.get('best_score', -1.)
            hist       = ck.get('history', [])
            start_ep   = len(hist) + 1
            print(f"  Resuming from ep{start_ep-1} (best={best_score:.4f})")
        except Exception as e:
            print(f"  Could not resume ({e}) — training from scratch")
        if start_ep > epochs:
            print(f"  Already complete. Returning existing checkpoint.")
            return best_ckpt, best_score

    for ep in range(start_ep, epochs + 1):
        t0        = time.time()
        using_swa = ep >= SWA_START

        if ep == SWA_START:
            for g in opt.param_groups: g['lr'] = SWA_LR
            print(f"  [ep{ep}] ─── SWA started (lr={SWA_LR}) ───")

        tr_loss = train_epoch(model, ds_tr, opt,
                               None if using_swa else sched, scaler, combo_w)
        if using_swa:
            swa_model.update_parameters(model)
            swa_s.step()

        # Single-pass val during training for speed; TTA only at final eval
        cp, sp, pp, fns = val_epoch(model, ds_va, tta_passes=1)
        pm              = decode_with_presence(cp, sp, pp, fns)
        ap, asal, score = official_scores(pm, ds_va.df)

        tag = "[SWA] " if using_swa else "      "
        print(f"  ep{ep:3d}/{epochs} {tag}loss={tr_loss:.4f}  "
              f"pres={ap:.4f}  sal={asal:.4f}  avg={score:.4f}  [{time.time()-t0:.0f}s]")

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            print("    ↑ new best!")

        hist.append(dict(ep=ep, loss=tr_loss, pres=ap, sal=asal, avg=score))

    # ── SWA final evaluation with TTA ────────────────────────────────────
    if SWA_START <= epochs:
        print("  Updating SWA batch-norm stats ...")
        update_bn(make_loader(ds_tr, shuffle=True), swa_model, device=DEVICE)
        # Use TTA for the definitive SWA evaluation
        cp_s, sp_s, pp_s, fns_s = val_epoch(swa_model.module, ds_va,
                                              tta_passes=TTA_PASSES)
        pm_s                     = decode_with_presence(cp_s, sp_s, pp_s, fns_s)
        ap_s, asal_s, sc_s       = official_scores(pm_s, ds_va.df)
        print(f"  SWA+TTA: pres={ap_s:.4f}  sal={asal_s:.4f}  avg={sc_s:.4f}")
        if sc_s > best_score:
            best_score = sc_s
            best_state = copy.deepcopy(swa_model.module.state_dict())
            print("    ↑ SWA+TTA is best!")

    torch.save(dict(
        model_state=best_state, stats=stats,
        modalities=selected_mods, cfg=cfg,
        best_score=best_score, history=hist
    ), best_ckpt)
    print(f"  ✓ Saved → {best_ckpt}  (score={best_score:.4f})")

    del model, swa_model
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    gc.collect()
    return best_ckpt, best_score


In [14]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 14 — Train all variants × all folds                   ║
# ║  6 variants × 5 folds = 30 models; top-16 go to ensemble    ║
# ╚══════════════════════════════════════════════════════════════╝
best_ckpts = []
for cfg in MODEL_VARIANTS:
    for f in FOLDS_TO_TRAIN:
        path, score = run_fold(f, cfg, meta_df, stats, train_cache)
        if path:
            best_ckpts.append(dict(model=cfg['name'], fold=f, ckpt=path, score=score))

best_ckpts_sorted = sorted(best_ckpts, key=lambda x: x['score'], reverse=True)
print("\nAll checkpoints (sorted):")
for b in best_ckpts_sorted:
    print(f"  {b['model']:12s}  fold={b['fold']}  score={b['score']:.4f}")



────────────────────────────────────────────────────────────
Fold 0 | cmaf_s
────────────────────────────────────────────────────────────
  [train fold=0] n=1934  feat_len=8448
  [val fold=0] n=522  feat_len=8448
  Params: 1.254M
  Resuming from ep90 (best=0.2634)
  Already complete. Returning existing checkpoint.

────────────────────────────────────────────────────────────
Fold 1 | cmaf_s
────────────────────────────────────────────────────────────
  [train fold=1] n=1970  feat_len=8448
  [val fold=1] n=486  feat_len=8448
  Params: 1.254M
  Resuming from ep90 (best=0.2508)
  Already complete. Returning existing checkpoint.

────────────────────────────────────────────────────────────
Fold 2 | cmaf_s
────────────────────────────────────────────────────────────
  [train fold=2] n=1988  feat_len=8448
  [val fold=2] n=468  feat_len=8448
  Params: 1.254M
  Resuming from ep90 (best=0.2569)
  Already complete. Returning existing checkpoint.

────────────────────────────────────────────────

In [15]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 15 — Inference & ensemble helpers                     ║
# ║  NEW v2: TTA in predict_one_model; presence-guided decoding  ║
# ╚══════════════════════════════════════════════════════════════╝
def load_model_ckpt(path):
    ck  = torch.load(path, map_location=DEVICE, weights_only=False)
    cfg = ck['cfg']
    m   = CMAFNet(ck['stats'], ck['modalities'],
                   mod_proj=cfg['mod_proj'], n_heads=cfg['n_heads'],
                   n_layers=cfg['n_layers'], dropout=cfg['dropout'])
    m.load_state_dict(ck['model_state'], strict=False)  # strict=False for v1→v2 compat
    m.to(DEVICE).eval()
    return m


@torch.no_grad()
def predict_one_model(model, loader,
                      tta_passes: int = TTA_PASSES,
                      tta_noise: float = 0.015):
    """
    TTA inference (NEW in v2).

    Pass 0: clean features (no noise) — preserves the original signal.
    Passes 1‥N-1: features + small Gaussian noise — encourages the model
    to produce more calibrated probability distributions by averaging
    across slightly different views of the same sample.

    tta_noise=0.015 is on the scale of the z-scored feature std (≈1),
    so it introduces ~1.5% perturbation — enough to diversify predictions
    without corrupting semantics.
    """
    cp, sp, pp, fns = [], [], [], []
    for batch in loader:
        feats    = batch['features'].to(DEVICE, non_blocking=True)
        batch_cp = batch_sp = batch_pp = None
        for t in range(tta_passes):
            noisy = feats if t == 0 else feats + torch.randn_like(feats) * tta_noise
            with get_amp():
                cl, sl, pl = model(noisy)
            bc = torch.softmax(cl, dim=1)
            bs = torch.softmax(sl, dim=1)
            bp = torch.sigmoid(pl)
            batch_cp = bc if batch_cp is None else batch_cp + bc
            batch_sp = bs if batch_sp is None else batch_sp + bs
            batch_pp = bp if batch_pp is None else batch_pp + bp
        cp.append((batch_cp / tta_passes).cpu().numpy())
        sp.append((batch_sp / tta_passes).cpu().numpy())
        pp.append((batch_pp / tta_passes).cpu().numpy())
        fns.extend(batch['fname'])
    return np.vstack(cp), np.vstack(sp), np.vstack(pp), fns


def ensemble_predict(items, ds):
    """
    Score²-weighted average ensemble including presence probs.
    Creates a FRESH DataLoader per model — avoids iterator exhaustion.
    """
    w = np.array([x['score'] ** 2 for x in items], dtype=np.float64)
    w /= w.sum()
    cs = ss = ps = fns_ref = None
    for item, wi in zip(items, w):
        model  = load_model_ckpt(item['ckpt'])
        loader = make_loader(ds, shuffle=False)
        cp, sp, pp, fns = predict_one_model(model, loader)
        del model
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
        if fns_ref is None:
            fns_ref = fns
            cs = cp * wi; ss = sp * wi; ps = pp * wi
        else:
            cs += cp * wi; ss += sp * wi; ps += pp * wi
    return cs, ss, ps, fns_ref


In [16]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 16 — True out-of-fold CV validation                   ║
# ╚══════════════════════════════════════════════════════════════╝
def cv_validation(all_ckpts, meta_df, stats, ram_cache):
    fold_results = []
    for fold in sorted(meta_df['fold'].unique()):
        oof = sorted([x for x in all_ckpts if x['fold'] == fold],
                     key=lambda x: x['score'], reverse=True)
        if not oof:
            print(f"  Fold {fold}: no checkpoints — skipping")
            continue
        ds_va = BlEmoDataset(meta_df, selected_mods, fold, 'val',
                              stats, ram_cache, 0., 0.)
        cs, ss, ps, fns = ensemble_predict(oof, ds_va)
        pm              = decode_with_presence(cs, ss, ps, fns)
        ap, asal, avg   = official_scores(pm, ds_va.df)
        names = [x['model'] for x in oof]
        print(f"  Fold {fold} [{len(oof)} models: {', '.join(names)}]")
        print(f"    pres={ap:.4f}  sal={asal:.4f}  avg={avg:.4f}")
        fold_results.append((ap, asal, avg))

    if not fold_results:
        return 0., 0., 0.

    mean_p   = float(np.mean([r[0] for r in fold_results]))
    mean_sal = float(np.mean([r[1] for r in fold_results]))
    mean_avg = float(np.mean([r[2] for r in fold_results]))

    print(f"\n{'═'*55}")
    print(f"CV MEAN (true OOF)   pres={mean_p:.4f}  sal={mean_sal:.4f}  avg={mean_avg:.4f}")
    print(f"\nBaseline reference:")
    print(f"  ImageBind+WavLM    val pres=0.345  sal=0.170  avg=0.258")
    print(f"  VideoMAEv2+HuBERT  val pres=0.332  sal=0.138  avg=0.235")
    print(f"  HiCMAE             val pres=0.298  sal=0.180  avg=0.239")
    print(f"  Trivial baseline   val pres=0.077  sal=0.000  avg=0.039")
    if mean_p   > 0.345: print(f"\n  ✅ BEATS presence baseline (ImageBind+WavLM 0.345)!")
    if mean_sal > 0.180: print(f"  ✅ BEATS salience baseline (HiCMAE 0.180)!")
    if mean_avg > 0.258: print(f"  ✅ BEATS avg baseline (ImageBind+WavLM 0.258)!")
    return mean_p, mean_sal, mean_avg


cv_validation(best_ckpts, meta_df, stats, train_cache)


  [val fold=0] n=522  feat_len=8448
  Fold 0 [6 models: cmaf_xxl, cmaf_m, cmaf_xl, cmaf_deep, cmaf_l, cmaf_s]
    pres=0.4061  sal=0.1643  avg=0.2852
  [val fold=1] n=486  feat_len=8448
  Fold 1 [6 models: cmaf_m, cmaf_xxl, cmaf_xl, cmaf_deep, cmaf_s, cmaf_l]
    pres=0.3951  sal=0.1333  avg=0.2642
  [val fold=2] n=468  feat_len=8448
  Fold 2 [6 models: cmaf_m, cmaf_deep, cmaf_xl, cmaf_xxl, cmaf_l, cmaf_s]
    pres=0.4124  sal=0.1208  avg=0.2666
  [val fold=3] n=472  feat_len=8448
  Fold 3 [6 models: cmaf_m, cmaf_deep, cmaf_xxl, cmaf_s, cmaf_xl, cmaf_l]
    pres=0.4110  sal=0.1417  avg=0.2763
  [val fold=4] n=508  feat_len=8448
  Fold 4 [6 models: cmaf_m, cmaf_l, cmaf_xxl, cmaf_s, cmaf_deep, cmaf_xl]
    pres=0.3386  sal=0.0912  avg=0.2149

═══════════════════════════════════════════════════════
CV MEAN (true OOF)   pres=0.3926  sal=0.1303  avg=0.2615

Baseline reference:
  ImageBind+WavLM    val pres=0.345  sal=0.170  avg=0.258
  VideoMAEv2+HuBERT  val pres=0.332  sal=0.138  avg=0.235

(0.39263695706107093, 0.13026705276705278, 0.2614520049140618)

In [17]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 17 — Test submission (ensemble of top-K models)       ║
# ║  Uses TTA + presence-guided decoding                        ║
# ╚══════════════════════════════════════════════════════════════╝
test_ensemble = best_ckpts_sorted[:ENSEMBLE_TOPK]
print(f"Test ensemble: {len(test_ensemble)} models")
for b in test_ensemble:
    print(f"  {b['model']:12s}  fold={b['fold']}  score={b['score']:.4f}")

test_ds         = BlEmoTestDataset(test_samples, selected_mods, stats, test_cache)
cs, ss, ps, fns = ensemble_predict(test_ensemble, test_ds)
pred_map        = decode_with_presence(cs, ss, ps, fns)

out = f"{OUTPUT_DIR}/submission.json"
with open(out, 'w') as f:
    json.dump({'predictions': pred_map}, f, indent=2)
zp = out.replace('.json', '.zip')
with zipfile.ZipFile(zp, 'w') as z:
    z.write(out, os.path.basename(out))

print(f"\nSubmission files:")
print(f"  JSON: {out}")
print(f"  ZIP:  {zp}")
print(f"  Total predictions: {len(pred_map)}")
sample_key = list(pred_map.keys())[0]
print(f"\nSample: {sample_key} -> {pred_map[sample_key]}")


Test ensemble: 16 models
  cmaf_xxl      fold=0  score=0.2876
  cmaf_m        fold=0  score=0.2846
  cmaf_m        fold=3  score=0.2817
  cmaf_xl       fold=0  score=0.2784
  cmaf_m        fold=2  score=0.2771
  cmaf_m        fold=1  score=0.2743
  cmaf_deep     fold=0  score=0.2721
  cmaf_l        fold=0  score=0.2716
  cmaf_deep     fold=3  score=0.2678
  cmaf_deep     fold=2  score=0.2673
  cmaf_xxl      fold=3  score=0.2670
  cmaf_xxl      fold=1  score=0.2652
  cmaf_xl       fold=2  score=0.2652
  cmaf_s        fold=3  score=0.2645
  cmaf_xxl      fold=2  score=0.2643
  cmaf_s        fold=0  score=0.2634

Submission files:
  JSON: /content/drive/MyDrive/blemore_project-backup/blemore_output-cmaf-02/submission.json
  ZIP:  /content/drive/MyDrive/blemore_project-backup/blemore_output-cmaf-02/submission.zip
  Total predictions: 594

Sample: test_0001.mov -> [{'emotion': 'ang', 'salience': 30.0}, {'emotion': 'disg', 'salience': 70.0}]
